# E · Direct retrieval: fidelity, coverage và căn cứ ảnh

Notebook CPU đọc lượt `path-*` có `explanation_mode=object`. Bootstrap theo query, giữ seed trong cụm; coverage gồm cả query không có cạnh đối tượng phù hợp. Fidelity so với cạnh `contains` khác trên cùng ảnh đích khi có đối chứng độc lập. Chấm ảnh riêng để biết đối tượng có thật; score drop không phải validity ngữ nghĩa của toàn caption. `failure` trong benchmark cũ chỉ nghĩa là top-1 khác ID GT; ảnh đó vẫn có thể phù hợp với query. Các output lưu trong notebook từ lượt cũ cần chạy lại sau thay đổi này.


In [ ]:
from pathlib import Path
import json,sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
REPO=Path('/content/drive/MyDrive/graphml-project3') if Path('/content/drive/MyDrive/graphml-project3').is_dir() else next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'src/mmxai/xai/xai-experiment').is_dir())
EXP=REPO/'src/mmxai/xai/xai-experiment'
sys.path[:0]=[str(EXP),str(REPO/'src')]
from xai_experiment.paths import runs_dir, latest_artifact
RUNS=runs_dir()
from xai_experiment.review_workbook import create_review_workbook
from xai_experiment.image_download import fetch_review_image
RESULTS_DIR=None  # set an explicit completed path-* directory for the report
if RESULTS_DIR is None:
    complete=sorted(p.parent for p in RUNS.glob('path-*/complete.json')
                    if json.loads((p.parent/'protocol.json').read_text()).get('explanation_mode')=='object')
    assert complete,'Run notebook 05 in object mode first.'
    RESULTS_DIR=complete[-1]
RESULTS_DIR=Path(RESULTS_DIR)
assert json.loads((RESULTS_DIR/'protocol.json').read_text()).get('explanation_mode')=='object'
results=pd.read_csv(RESULTS_DIR/'xai_results.csv')
cases=pd.read_csv(RESULTS_DIR/'case_manifest.csv')
cases['exact_id_outcome']=np.where(cases.gt_image_id.astype(str)==cases.retrieved_image_id.astype(str),'match','mismatch')
if 'retrieval_outcome' in cases:
    legacy=np.where(cases.retrieval_outcome=='success','match','mismatch')
    assert (legacy==cases.exact_id_outcome).all(), 'Legacy success/failure must mean exact GT ID match only'
paths=[json.loads(line) for line in (RESULTS_DIR/'paths.jsonl').open(encoding='utf8')]
meta=json.loads((RESULTS_DIR/'complete.json').read_text())
assert len(results)==meta['n_result_rows'] and len(paths)==meta['n_path_rows']
analysis=RESULTS_DIR/'analysis'; analysis.mkdir(exist_ok=True)
print(RESULTS_DIR,len(results),len(cases),len(paths))


## 1 · Mẫu số, object yield và đối chứng

Mỗi `dataset × budget` giữ toàn bộ case đã đăng ký. `exact_id_outcome` chỉ so ID top-1 với GT và không đo mức phù hợp ngữ nghĩa. Sparsity = 1 − số cạnh explanation duy nhất / `E_eligible_contains`; báo thêm số cạnh `contains` đủ điều kiện. Random control chỉ lấy cạnh `contains` khác trên chính ảnh đích. Thiếu cạnh đối chứng thì fidelity advantage là N/A, không loại case khỏi coverage.


In [ ]:
main=results[results.intervention=='object_path'].copy()
assert not main.duplicated(['dataset','query_id','model_seed','screen_path_limit','budget']).any()
assert (main.n_screened_contains<=main.n_eligible_explanation).all() and (main.n_removed_unique<=main.budget).all()
main['matched_advantage']=main.final_score_drop-main.random_final_drop_mean
coverage=main.groupby(['dataset','screen_path_limit','budget']).agg(n_cases=('query_id','size'),n_queries=('query_id','nunique'),positive_object_rate=('has_positive_object_path','mean'),selected_path_rate=('n_paths',lambda x:(x>0).mean()),no_explanation_rate=('status',lambda x:(x!='ok').mean()),structural_no_path_rate=('status',lambda x:x.isin(['no_object_path','no_path']).mean()),mean_eligible=('n_eligible_explanation','mean'),mean_screen_coverage=('screen_coverage','mean'),mean_sparsity=('sparsity','mean'),matched_cases=('random_n',lambda x:(x>0).sum()),unique_random_sets=('random_unique_n','sum'),mean_fidelity_advantage=('matched_advantage','mean')).reset_index()
display(coverage)
exact_id_summary=cases.groupby('dataset').agg(n_rows=('query_id','size'),n_queries=('query_id','nunique'),exact_id_match_rate=('exact_id_outcome',lambda x:(x=='match').mean())).reset_index()
display(exact_id_summary)
display(main.groupby(['dataset','screen_path_limit','budget']).status.value_counts())


## 2 · CI 95% theo query và hình fidelity–sparsity–coverage

Lấy lại query ID, không lấy lại từng dòng seed. CI fidelity chỉ tính trên query có random match và kèm `n_matched_queries`. Yield/coverage dùng toàn bộ query. Không so trực tiếp thang fuse VG và PSG.


In [ ]:
rng=np.random.default_rng(20261121); ci=[]
for key,part in main.groupby(['dataset','screen_path_limit','budget']):
    by_query=part.groupby('query_id').agg(coverage=('n_paths',lambda x:(x>0).mean()),advantage=('matched_advantage','mean'))
    out={'dataset':key[0],'screen_path_limit':key[1],'budget':key[2],'n_queries':len(by_query),'n_matched_queries':by_query.advantage.notna().sum()}
    for field in ('coverage','advantage'):
        values=by_query[field].dropna().to_numpy(); out[field+'_mean']=float(values.mean()) if len(values) else np.nan
        if len(values):
            draws=rng.choice(values,size=(5000,len(values)),replace=True).mean(axis=1)
            out[field+'_ci_low'],out[field+'_ci_high']=map(float,np.quantile(draws,[.025,.975]))
        else: out[field+'_ci_low']=out[field+'_ci_high']=np.nan
    ci.append(out)
ci=pd.DataFrame(ci); display(ci)
coverage.to_csv(analysis/'coverage.csv',index=False); ci.to_csv(analysis/'cluster_bootstrap_ci.csv',index=False)
fig,axes=plt.subplots(1,3,figsize=(15,4))
for (dataset,limit),sub in coverage.groupby(['dataset','screen_path_limit']):
    sub=sub.sort_values('budget'); label=f'{dataset.upper()} screen={limit}'
    for ax,y in zip(axes,['mean_fidelity_advantage','mean_sparsity','selected_path_rate']): ax.plot(sub.budget,sub[y],marker='o',label=label)
for ax,title in zip(axes,['Matched final drop advantage','Sparsity / target contains edges','Object evidence yield / all cases']):
    ax.set_title(title); ax.set_xlabel('Unique-edge budget'); ax.grid(alpha=.3)
axes[0].legend(fontsize=7); fig.tight_layout(); fig.savefig(analysis/'fidelity_sparsity_coverage.png',dpi=160)
display(fig); plt.close(fig)


## 3 · Tối đa 10 case study và Excel chấm ảnh

Chọn 5 GT-ID match/5 GT-ID mismatch theo seed 0, gồm cả case không có cạnh đối tượng giải thích. Nhúng GT và top-1 vào workbook khi tải được. Ở tab Case review, chấm riêng `retrieved_relevance` cho ảnh top-1 khác ID GT, ghi `visual_reason` và `reviewer`. Phần bằng chứng đối tượng có thể chấm trên một workbook; cạnh `contains` không tự xác minh hành động/vị trí của cả caption.


In [ ]:
limit=int(cases.screen_path_limit.max()); pool=cases[(cases.model_seed==0)&(cases.screen_path_limit==limit)]
def choose_outcome(outcome):
    part=pool[pool.exact_id_outcome==outcome].sort_values(['dataset','query_id'])
    preferred=pd.concat([x.head(2) for _,x in part.groupby('dataset')]) if len(part) else part
    remaining=part.drop(preferred.index).sort_values(['query_id','dataset'])
    return pd.concat([preferred,remaining.head(max(0,5-len(preferred)))]).head(5)
chosen=pd.concat([choose_outcome('match'),choose_outcome('mismatch')])
print('Case studies:',len(chosen),'| GT-ID match:',(chosen.exact_id_outcome=='match').sum(),'| GT-ID mismatch:',(chosen.exact_id_outcome=='mismatch').sum())
if len(chosen)<10: print('Insufficient GT-ID match/mismatch cases; reporting the available cases only')
image_dir=analysis/'case_images'; image_dir.mkdir(exist_ok=True)
ALLOW_UNVERIFIED_TLS=True  # public dataset images only; provenance is saved below
path_index={}
for p in paths:
    if p.get('model_seed')==0 and p.get('screen_path_limit')==limit: path_index.setdefault((p['dataset'],str(p['query_id'])),[]).append(p)
verified=pd.read_csv(EXP/'registry/queries_direct_verified.csv',keep_default_na=False,dtype={'query_id':str})
gt_labels={(r['dataset'],r['query_id']):r['gt_relevance'] for r in verified.to_dict('records')}
review_cases=[]; review_steps=[]; download_records=[]
for row in chosen.to_dict('records'):
    row['gt_relevance']=gt_labels.get((row['dataset'],str(row['query_id'])),'')
    row['retrieved_relevance']='supported' if row['exact_id_outcome']=='match' and row['gt_relevance']=='supported' else ''
    row['retrieval_outcome']='GT ID match' if row['exact_id_outcome']=='match' else 'GT ID mismatch'
    ps=[p for p in path_index.get((row['dataset'],str(row['query_id'])),[]) if p.get('level')=='object' and 2 in p.get('selected_at_budgets',[])]
    row['explanation_level']='object_evidence' if ps else 'no_object_explanation'
    row['path_or_edge_text']='; '.join(' → '.join(p['node_ids']) for p in ps) or 'no path selected'
    for role,url_key,file_key in [('gt','gt_source_url','gt_image_file'),('retrieved','retrieved_source_url','retrieved_image_file')]:
        path=image_dir/f"{row['dataset']}_{row['query_id']}_{role}.jpg"
        if path.is_file():
            row[file_key]=str(path)
            download_records.append({'dataset':row['dataset'],'query_id':row['query_id'],'role':role,'status':'cached'})
            continue
        try:
            info=fetch_review_image(row[url_key],path,allow_unverified_tls=ALLOW_UNVERIFIED_TLS)
            row[file_key]=str(path)
            download_records.append({'dataset':row['dataset'],'query_id':row['query_id'],'role':role,**info})
        except Exception as exc:
            row[file_key]=''; download_records.append({'dataset':row['dataset'],'query_id':row['query_id'],'role':role,'source_url':row[url_key],'status':'failed','error':str(exc)})
            print('Image unavailable:',row['query_id'],role,str(exc)[:180])
    review_cases.append(row)
    if not ps: review_steps.append({'dataset':row['dataset'],'query_id':row['query_id'],'target_image_id':row['retrieved_image_id'],'path_id':'no_path'})
    for p in ps:
        for i,e in enumerate(p['edge_ids']):
            review_steps.append({'dataset':row['dataset'],'query_id':row['query_id'],'target_image_id':row['retrieved_image_id'],'path_id':p['path_id'],'step_index':i+1,'edge_id':e,'source_node':p['node_ids'][i],'target_node':p['node_ids'][i+1],'predicate_hint':p['predicates'][i],'edge_type':p['edge_types'][i],'edge_provenance':p['sources'][i],'model_score_drop':p['final_score_drop']})
workbook=analysis/'validity_case_review_10.xlsx'
if workbook.exists():
    existing=pd.read_excel(workbook,sheet_name='Case review',keep_default_na=False,dtype={'query_id':str})
    expected_ids={(r['dataset'],str(r['query_id'])) for r in review_cases}
    actual_ids={(r['dataset'],str(r['query_id'])) for r in existing.to_dict('records')}
    assert len(existing)==len(review_cases) and actual_ids==expected_ids, 'Existing workbook has a different case set'
    print('Keeping existing review workbook and its annotations:',workbook)
else:
    create_review_workbook(review_cases,review_steps,workbook,analysis)
pd.DataFrame(download_records).to_csv(analysis/'image_download_manifest.csv',index=False)
print('Downloaded/available images:',sum(r['status']!='failed' for r in download_records),'/',len(download_records))
annotations=pd.DataFrame(review_steps)
for col in ['reviewer','object_supported','visual_evidence','step_decision','support_source','evidence_reason']:
    annotations[col]=''
annotations_path=analysis/'validity_annotations.csv'
if not annotations_path.exists(): annotations.to_csv(annotations_path,index=False)
pd.DataFrame(review_cases).to_csv(analysis/'case_studies_10.csv',index=False)
print('Review workbook:',workbook,'step rows:',len(review_steps))


## 4 · Chấm mức phù hợp của ảnh top-1

Một ảnh top-1 **khác ID GT** vẫn có thể hỗ trợ toàn query. Trong workbook `validity_case_review_10.xlsx`, tab **Case review**, các ô `retrieved_relevance` (`supported`/`contradicted`/`uncertain`), `visual_reason` và `reviewer` có thể lưu nhãn từng dòng. Người dùng đã xác nhận bằng mắt rằng cả năm mismatch của lượt `path-20261121-184857` hỗ trợ query; cell dưới dùng xác nhận này cho ô còn trống, nhưng nhãn đã điền trong workbook được ưu tiên. Chạy lại cell sau khi lưu Excel nếu có sửa nhãn. Tỷ lệ thu được chỉ mô tả **10 case được chọn cân bằng theo ID**, không phải semantic@1 của toàn bộ 34 query.


In [ ]:
RELEVANCE_WORKBOOK=analysis/'validity_case_review_10.xlsx'  # dùng workbook đã điền; có thể đặt đường dẫn khác
reviewed=pd.read_excel(RELEVANCE_WORKBOOK,sheet_name='Case review',keep_default_na=False,dtype={'query_id':str})
reviewed['query_id']=reviewed.query_id.astype(str)
expected=chosen[['dataset','query_id','exact_id_outcome']].copy()
expected['query_id']=expected.query_id.astype(str)
fields=['dataset','query_id','retrieved_relevance','visual_reason','reviewer']
assert set(fields) <= set(reviewed.columns)
semantic=expected.merge(reviewed[fields],on=['dataset','query_id'],how='left',validate='one_to_one')
assert len(semantic)==len(chosen) and semantic.retrieved_relevance.notna().all()
semantic['relevance_source']='case_review_workbook'
match_without_label=(semantic.exact_id_outcome=='match') & semantic.retrieved_relevance.eq('')
semantic.loc[match_without_label,'retrieved_relevance']='supported'  # same verified GT image
semantic.loc[match_without_label,'relevance_source']='verified_gt_same_image'
USER_CONFIRMED_MISMATCH_RUN='path-20261121-184857'
if RESULTS_DIR.name==USER_CONFIRMED_MISMATCH_RUN:
    confirmed=(semantic.exact_id_outcome=='mismatch') & semantic.retrieved_relevance.eq('')
    semantic.loc[confirmed,'retrieved_relevance']='supported'
    semantic.loc[confirmed,'relevance_source']='user_visual_confirmation'
    supported_mismatch=(semantic.exact_id_outcome=='mismatch') & semantic.retrieved_relevance.eq('supported')
    auto_provenance=supported_mismatch & (semantic.visual_reason.eq('') | semantic.reviewer.eq(''))
    semantic.loc[auto_provenance,'relevance_source']='user_visual_confirmation'
    semantic.loc[supported_mismatch & semantic.visual_reason.eq(''),'visual_reason']='User visually confirmed retrieved image supports the full query in the case workbook.'
    semantic.loc[supported_mismatch & semantic.reviewer.eq(''),'reviewer']='user_visual_confirmation'
missing=semantic[(semantic.exact_id_outcome=='mismatch') & (
    ~semantic.retrieved_relevance.isin(['supported','contradicted','uncertain']) |
    semantic.visual_reason.eq('') | semantic.reviewer.eq(''))]
if len(missing):
    print('Cần chấm ảnh top-1 khác ID GT cho các query:')
    display(missing[['dataset','query_id','retrieved_relevance','visual_reason','reviewer']])
else:
    assert semantic.loc[semantic.exact_id_outcome=='match','retrieved_relevance'].eq('supported').all()
    semantic.to_csv(analysis/'semantic_case_review.csv',index=False)
    counts=semantic.retrieved_relevance.value_counts().reindex(['supported','contradicted','uncertain'],fill_value=0)
    text=f"""# Top-1 relevance on stratified case studies

GT-ID match: {(semantic.exact_id_outcome=='match').sum()}/{len(semantic)}.
GT-ID mismatch: {(semantic.exact_id_outcome=='mismatch').sum()}/{len(semantic)}.
Top-1 supported: {counts['supported']}/{len(semantic)}; contradicted: {counts['contradicted']}; uncertain: {counts['uncertain']}.

These ten cases were selected to balance exact-ID match/mismatch. Blank mismatch labels for run path-20261121-184857 were filled from the user's visual confirmation; explicit workbook labels take precedence. This is not a semantic accuracy estimate for all 34 queries. Exact-ID ranking metrics remain valid for the dataset benchmark.
"""
    (analysis/'semantic_case_review.md').write_text(text,encoding='utf8')
    display(semantic)
    print(text)


## 5 · Bằng chứng đối tượng (một workbook, tùy chọn)

Nếu muốn báo object/visual support của cạnh `contains`, điền `object_supported`, `visual_evidence`, `step_decision`, `evidence_reason` và `reviewer` trong tab **Step review**. Cell dưới chỉ tính trên nhãn đã điền; không tính đồng thuận giữa người chấm và không yêu cầu bản hòa giải.


In [ ]:
STEP_REVIEW_WORKBOOK=analysis/'validity_case_review_10.xlsx'
step_review=pd.read_excel(STEP_REVIEW_WORKBOOK,sheet_name='Step review',keep_default_na=False,dtype={'query_id':str})
real=step_review[step_review.edge_id.astype(str).str.strip().ne('')].copy()
labels=['supported','contradicted','uncertain']
judgements=['object_supported','visual_evidence','step_decision']
ready=(len(real)>0 and all(real[field].isin(labels).all() for field in judgements)
       and real.reviewer.astype(str).str.strip().ne('').all()
       and real.evidence_reason.astype(str).str.strip().ne('').all())
if not ready:
    print('Object/visual support chưa có đủ nhãn trong Step review; báo cáo fidelity và semantic relevance vẫn hợp lệ.')
else:
    real['object_step_supported']=real[['object_supported','step_decision']].eq('supported').all(axis=1)
    real['visual_step_supported']=real[['visual_evidence','step_decision']].eq('supported').all(axis=1)
    real['step_known']=real.step_decision.ne('uncertain')
    paths_review=real.groupby(['dataset','query_id','path_id']).agg(
        object_supported=('object_step_supported','all'),
        visual_supported=('visual_step_supported','all'),
        known=('step_known','all')).reset_index()
    real.to_csv(analysis/'object_step_review.csv',index=False)
    paths_review.to_csv(analysis/'object_path_review.csv',index=False)
    summary=f"""# Object evidence on selected case studies

Reviewed steps: {len(real)}; object step support: {real.object_step_supported.mean():.3f}; visual step support: {real.visual_step_supported.mean():.3f}.
Reviewed paths: {len(paths_review)}; object path support: {paths_review.object_supported.mean():.3f}; visual path support: {paths_review.visual_supported.mean():.3f}.
These labels concern object evidence only; a contains edge does not verify the full caption.
"""
    (analysis/'object_evidence_review.md').write_text(summary,encoding='utf8')
    print(summary)
